# Tutorial: creación de ETLs con PySpark

**Nombre: Daniel Felipe Pérez Jaimes**

**Codigo estudiante: 202621404**

## 1. Introducción	
    ¿Qué aprenderá?
	En este tutorial aprenderá cómo puede utilizar PySpark para crear un proceso de ETL básico.

	¿Qué construirá?     
        Construirá un ETL que toma los datos desde la base de datos transacional de WideWorldImporters (WWImportersTransactional), los transforma  a una representación cercana al análisis y los  almacena en la base de datos relacional WWImportersDWH.
    
	¿Para qué?
	La construcción de ETLs que se ajusten a modelos multidimensionales es un paso necesario dentro de un proceso de analìtica 1.0 , pues permite tomar los datos crudos de una fuente, generalmente transaccional, para transformarlos en datos limpios que puedan utilizarse para la toma de decisiones.
    
    ¿Qué necesita?
    1. Python 3.13 y Java 21 (ya instalados en la máquina virtual)
    2. VS Code con la extensión Jupyter (kernel "Python 3 (ipykernel)")
    3. Paquetes: PySpark (4.2.0), pandas (2.3.3), numpy (2.5.3), pyarrow (25.0.1) y matplotlib (3.11.2)
    4. Controlador Connector J MySQL 9.7.0 (el notebook lo descarga automáticamente la primera vez)
    5. Servidor SQL con base de datos relacional "WWImportersTransactional" y base de datos relacional que corresponde a la bodega de WWI "DB_<periodo>_<usuario>"
	

## 2. Proceso de ETL para una dimensión.

En este proceso de ETL, se extraen los datos de las **órdenes de compra** de una base de datos transaccional y se almacenan en otra base de datos que corresponde a la bodega de datos, siguiendo una aproximación ROLAP. A continuación, se presenta el modelo multidimensional que es el modelo conceptual que representa el proceso de registro de órdenes de compra. Este modelo se utilizó para crear las tablas en la bodega de datos que representan el proceso de negocio y que serán cargadas como resultado del proceso ETL. 

Tenga en cuenta que las llaves ID_XXXX presentes en el modelo hacen referencia a las llaves de la bodega. Por otra parte, en el proceso de ETL se van a tener en cuenta las llaves transaccionales (**WWImportersTransactional**). La nomenclatura para utilizar es:

1.   ID_XXXX_DWH, para las llaves de la bodega.
2.   ID_XXXX_T, para las llaves transaccionales.


![Modelo ordenes](./WWI_modelo_ordenes.png)

El proceso de ETL debe ser diseñado antes de implementarse. A partir de las conclusiones del entendimiento de datos sabemos las fuentes que se van a  utilizar y la relación entre las fuentes. Adicionalmente, se cuenta con las respuestas de la organización a las preguntas, resultado del entendimiento de datos. De esa manera sabemos cómo se deben manipular los datos. 

Este proceso de ETL lo dividimos en seis bloques, uno para cada dimensión o <i>tabla de hechos</i> del modelo, con la única excepción de la dimensión de fecha que, por ser una dimensión especial que se genera de forma independiente, no se incluye aquí:

![ETL](./Disenio_ETL.png)

Recuerde que este es el diseño general. En el diseño completo se deben incluir las transformaciones realizadas a los datos a utilizarse en las dimensiones y tablas de hecho del modelo multidimensional, de acuerdo con lo que se muestra en la infografía de arquitectura de componentes (Componente proceso ETL) 

**Antes de ejecutar:** el cargue agrega filas (`mode('append')`). Si ejecuta el notebook de nuevo, borre antes las tablas que creó (`Empleados`, `Ciudad`, `TipoPaquete`, `Producto`, `Cliente`, `Hecho_Orden_Tmp`, `Hecho_Orden`).

In [1]:
# Configuración servidor base de datos transaccional
# Recuerde usar su usuario del curso y la contraseña asignada en el excel de conexión a maquina virtual como contraseña
db_user = 'DB_202615_df_perezj'
db_psswd = '202621404'

db_host = '157.253.236.120'
db_port = 8080
db_destino = 'DB_202615_df_perezj'      # su base de datos personal

if not db_user or not db_psswd or '<' in db_destino:
    raise ValueError('Complete db_user, db_psswd y db_destino antes de continuar')

jdbc_opciones = 'useConfigs=maxPerformance&connectTimeout=10000&socketTimeout=600000'

source_db_connection_string = f'jdbc:mysql://{db_host}:{db_port}/WWImportersTransactional?{jdbc_opciones}'

dest_db_connection_string = f'jdbc:mysql://{db_host}:{db_port}/{db_destino}?rewriteBatchedStatements=true&{jdbc_opciones}'

In [2]:
import os
import sys

# En la máquina virtual ya está todo instalado
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
IN_CODESPACES = os.environ.get('CODESPACES') == 'true'

if IN_COLAB or IN_CODESPACES:
    %pip install -q pyspark==4.2.0 pandas==2.3.3 numpy==2.5.3 pyarrow==25.0.1 matplotlib==3.11.2

if IN_COLAB:
    # Spark 4 requiere Java 17 o superior
    !apt-get install -y -qq openjdk-21-jdk-headless > /dev/null
    os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-21-openjdk-amd64'

ENVIRONMENT = 'colab' if IN_COLAB else 'codespaces' if IN_CODESPACES else 'local'
print(f'Entorno detectado: {ENVIRONMENT}')

from pyspark.sql import functions as f, SparkSession, types as t
from pyspark.sql.functions import udf, col, length, isnan, when, count, regexp_replace
from datetime import datetime

from pathlib import Path
import urllib.request

CONNECTOR_VERSION = '9.7.0'
CONNECTOR_URL = ('https://repo1.maven.org/maven2/com/mysql/mysql-connector-j/'
                 f'{CONNECTOR_VERSION}/mysql-connector-j-{CONNECTOR_VERSION}.jar')

def obtener_jar_mysql():
    env_path = os.environ.get('CONNECTOR_J_PATH', '')
    if env_path and Path(env_path).is_file():
        return Path(env_path)
    jar = Path.home() / '.miid4103' / 'jars' / f'mysql-connector-j-{CONNECTOR_VERSION}.jar'
    if not jar.exists():
        jar.parent.mkdir(parents=True, exist_ok=True)
        print(f'Descargando MySQL Connector/J {CONNECTOR_VERSION}...')
        parcial = jar.with_suffix('.part')
        urllib.request.urlretrieve(CONNECTOR_URL, parcial)
        parcial.replace(jar)
    return jar

MYSQL_JAR = obtener_jar_mysql()
MYSQL_JAR_URI = MYSQL_JAR.as_uri()
print(f'Conector JDBC: {MYSQL_JAR}')

ERROR: Ignored the following yanked versions: 2.4.0
ERROR: Ignored the following versions that require a different python version: 1.21.2 Requires-Python >=3.7,<3.11; 1.21.3 Requires-Python >=3.7,<3.11; 1.21.4 Requires-Python >=3.7,<3.11; 1.21.5 Requires-Python >=3.7,<3.11; 1.21.6 Requires-Python >=3.7,<3.11; 2.5.0 Requires-Python >=3.12; 2.5.0rc1 Requires-Python >=3.12; 2.5.1 Requires-Python >=3.12; 2.5.2 Requires-Python >=3.12; 2.5.3 Requires-Python >=3.12; 2.5.4 Requires-Python >=3.12
ERROR: Could not find a version that satisfies the requirement numpy==2.5.3 (from versions: 1.3.0, 1.4.1, 1.5.0, 1.5.1, 1.6.0, 1.6.1, 1.6.2, 1.7.0, 1.7.1, 1.7.2, 1.8.0, 1.8.1, 1.8.2, 1.9.0, 1.9.1, 1.9.2, 1.9.3, 1.10.0.post2, 1.10.1, 1.10.2, 1.10.4, 1.11.0, 1.11.1, 1.11.2, 1.11.3, 1.12.0, 1.12.1, 1.13.0, 1.13.1, 1.13.3, 1.14.0, 1.14.1, 1.14.2, 1.14.3, 1.14.4, 1.14.5, 1.14.6, 1.15.0, 1.15.1, 1.15.2, 1.15.3, 1.15.4, 1.16.0, 1.16.1, 1.16.2, 1.16.3, 1.16.4, 1.16.5, 1.16.6, 1.17.0, 1.17.1, 1.17.2, 1.17.3, 1.

In [3]:
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable
# Necesario en Windows si Python está en C:\Program Files
import pyspark
os.environ.setdefault('SPARK_HOME', os.path.dirname(pyspark.__file__))
# Spark 4.2 en Windows requiere HADOOP_HOME
if os.name == 'nt' and 'HADOOP_HOME' not in os.environ:
    hadoop_home = Path.home() / '.miid4103' / 'hadoop'
    (hadoop_home / 'bin').mkdir(parents=True, exist_ok=True)
    (hadoop_home / 'bin' / 'winutils.exe').touch()
    os.environ['HADOOP_HOME'] = str(hadoop_home)

spark = (SparkSession.builder
         .master('local[*]')
         .appName('MIID4103-NB04')
         .config('spark.driver.host', '127.0.0.1')
         .config('spark.jars', MYSQL_JAR_URI)
         .config('spark.sql.execution.arrow.pyspark.enabled', 'true')
         .config('spark.sql.shuffle.partitions', '8')
         .config('spark.ui.enabled', 'false')
         .config('spark.ui.showConsoleProgress', 'false')
         .config('spark.sql.ansi.enabled', 'false')    # comportamiento de Spark 3.5
         .getOrCreate())
spark.sparkContext.setLogLevel('ERROR')

Picked up JAVA_TOOL_OPTIONS: -Xss512k -XX:+UseContainerSupport
Picked up JAVA_TOOL_OPTIONS: -Xss512k -XX:+UseContainerSupport
26/11/04 04:10:18 WARN Utils: Your hostname, codespaces-7ef8dd resolves to a loopback address: 127.0.0.1; using 10.0.2.45 instead (on interface eth0)
26/11/04 04:10:18 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
26/11/04 04:10:18 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


### Conexión y carga de datos

Se define la función para conexión y cargue de dataframes desde la base de datos origen y luego la función para guardar un dataframe en una tabla de la base de datos destino.

In [4]:
def obterner_dataframe_desde_csv(_PATH, _sep):
    return spark.read.load(_PATH, format="csv", sep=_sep, inferSchema="true", header='true')

def obtener_dataframe_de_bd(db_connection_string, sql, db_user, db_psswd):
    df_bd = spark.read.format('jdbc')\
        .option('url', db_connection_string) \
        .option('dbtable', sql) \
        .option('user', db_user) \
        .option('password', db_psswd) \
        .option('driver', 'com.mysql.cj.jdbc.Driver') \
        .load()
    return df_bd.cache()

def guardar_db(db_connection_string, df, tabla, db_user, db_psswd):
    df.select('*').write.format('jdbc') \
      .mode('append') \
      .option('url', db_connection_string) \
      .option('dbtable', tabla) \
      .option('user', db_user) \
      .option('password', db_psswd) \
      .option('driver', 'com.mysql.cj.jdbc.Driver') \
      .option('batchsize', 10000) \
      .option('numPartitions', 4) \
      .save()

### BLOQUE 1
Empezamos con el bloque 1: la dimensión <i>Empleado</i>, su fuente de datos viene de la tabla transaccional <i>Personas</i>. En la sentencia SQL filtramos usando WHERE para seleccionar solo las personas que sean vendedores y recuperamos únicamente los atributos que queremos, de acuerdo con  modelo dimensional. Recuerde que también puede usar el **.select()** de pyspark si no conoce los atributos de las tablas transaccionales. Sin embargo, es más eficiente aplicar el filtro en la consulta, ya que no trae a memoria más información de la necesaria.

#### Extracción
A continuación, nos conectamos a la base de datos y extraemos la información deseada por medio de SQL, cargandola en un DataFrame PySpark, es decir en memoria. Note que aquí se pueden renombrar los atributos con la estructura <i>nombreActual AS nuevoNombre</i>. De la tabla de personas, En este paso, solo nos interesan los empleados, por lo cual se hace un filtro por medio del WHERE, buscando las personas cuyo atributo EsVendedor sea igual a 1.

In [5]:
sql_empleados = '''(SELECT DISTINCT ID_persona AS ID_Empleado, NombreCompleto AS Nombre, EsVendedor FROM WWImportersTransactional.Personas WHERE EsVendedor=1) AS Temp_empleados'''
empleados = obtener_dataframe_de_bd(source_db_connection_string, sql_empleados, db_user, db_psswd)
empleados.show(10)

+-----------+------------------+----------+
|ID_Empleado|            Nombre|EsVendedor|
+-----------+------------------+----------+
|          2|    Kayla Woodcock|      true|
|          3|     Hudson Onslow|      true|
|          6|     Sophia Hinton|      true|
|          7|         Amy Trefl|      true|
|          8|    Anthony Grosse|      true|
|         13|Hudson Hollinworth|      true|
|         14|         Lily Code|      true|
|         15|         Taj Shand|      true|
|         16|     Archer Lamble|      true|
|         20|       Jack Potter|      true|
+-----------+------------------+----------+



#### Transformación
Recuerde que, puede hacer uso de selectExpr, filter, where entre otras de PySpark para modificar los datos cargados. Por ejemplo, el siguiente código utiliza <i>selectExpr</i> para renombrar la columna ID_Empleado por ID_Empleado_T, esta es la convención que vamos a utilizar: "_T" para indicar que el ID es el que estaba en la base de datos transaccional y "_DWH" para indicar que son ID's propios de la bodega. Usamos withColumn y monotonicallu_increasing_id para crear un ID acumulativo para cada registro en el dataframe

In [6]:
# TRANSFORMACION
empleados = empleados.selectExpr('ID_Empleado as ID_Empleado_T','Nombre')
empleados = empleados.coalesce(1).withColumn('ID_Empleado_DWH', f.monotonically_increasing_id() + 1)
empleados = empleados.select('ID_Empleado_DWH','ID_Empleado_T','Nombre')
empleados.show(5)

+---------------+-------------+--------------+
|ID_Empleado_DWH|ID_Empleado_T|        Nombre|
+---------------+-------------+--------------+
|              1|            2|Kayla Woodcock|
|              2|            3| Hudson Onslow|
|              3|            6| Sophia Hinton|
|              4|            7|     Amy Trefl|
|              5|            8|Anthony Grosse|
+---------------+-------------+--------------+
only showing top 5 rows



#### Carga
Una vez realizado esto, se guardan los resultados en la base de datos destino

**OJO** Recuerde antes de guardar los datos que la tabla no exista o este vacía, para que no se guarden los mismos datos varias veces y no ocupar más espacio.

In [7]:
# CARGUE
guardar_db(dest_db_connection_string, empleados,f'{db_destino}.Empleados', db_user, db_psswd)

Verifique los resultados usando MySQL Workbench

### BLOQUE 2
Empezamos el bloque 2: dimensión ciudad. Su fuente de datos es una combinación de las tablas transaccionales <i>paises, provinciasEstados y ciudades</i>

#### Extracción

In [8]:
#EXTRACCION
sql_paises = '''(SELECT DISTINCT ID_Pais, Nombre, Continente, Region, Subregion FROM WWImportersTransactional.Paises) AS Temp_paises'''
sql_provincias_estados = '''(SELECT DISTINCT ID_EstadosProvincias AS ID_EstadoProvincia, NombreEstadoProvincia, TerritorioVentas, ID_Pais FROM WWImportersTransactional.EstadosProvincias) AS Temp_estados_provincias'''
sql_ciudades = '''(SELECT DISTINCT ID_ciudad as ID_ciudad_T, NombreCiudad, ID_EstadoProvincia, Poblacion FROM WWImportersTransactional.Ciudades) AS Temp_ciudades'''

paises = obtener_dataframe_de_bd(source_db_connection_string, sql_paises, db_user, db_psswd)
provincias_estados = obtener_dataframe_de_bd(source_db_connection_string, sql_provincias_estados, db_user, db_psswd)
ciudades = obtener_dataframe_de_bd(source_db_connection_string, sql_ciudades, db_user, db_psswd)

print(ciudades.columns, paises.columns, provincias_estados.columns)

['ID_ciudad_T', 'NombreCiudad', 'ID_EstadoProvincia', 'Poblacion'] ['ID_Pais', 'Nombre', 'Continente', 'Region', 'Subregion'] ['ID_EstadoProvincia', 'NombreEstadoProvincia', 'TerritorioVentas', 'ID_Pais']


#### Transformación

In [9]:
# TRANSFORMACION
ciudades = ciudades.join(provincias_estados, how = 'left', on = 'ID_EstadoProvincia')
ciudades = ciudades.join(paises, how = 'left', on = 'ID_Pais')
ciudades = ciudades.coalesce(1).withColumn('ID_Ciudad_DWH', f.monotonically_increasing_id() + 1)
ciudades = ciudades.select('ID_Ciudad_DWH','ID_ciudad_T','NombreCiudad','Continente','Nombre','Poblacion',
                          'Region','TerritorioVentas','NombreEstadoProvincia','Subregion') \
                    .withColumnRenamed('Nombre','Pais')
ciudades.show(5)

+-------------+-----------+------------+-------------+-------------+---------+--------+----------------+---------------------+----------------+
|ID_Ciudad_DWH|ID_ciudad_T|NombreCiudad|   Continente|         Pais|Poblacion|  Region|TerritorioVentas|NombreEstadoProvincia|       Subregion|
+-------------+-----------+------------+-------------+-------------+---------+--------+----------------+---------------------+----------------+
|            1|          1|  Aaronsburg|North America|United States|      613|Americas|         Mideast|         Pennsylvania|Northern America|
|            2|          3|      Abanda|North America|United States|      192|Americas|       Southeast|              Alabama|Northern America|
|            3|          4|   Abbeville|North America|United States|     5237|Americas|       Southeast|       South Carolina|Northern America|
|            4|          5|   Abbeville|North America|United States|     2908|Americas|       Southeast|              Georgia|Northern A

#### Carga

**OJO** Recuerde antes de guardar los datos que la tabla no exista o este vacía, para que no se guarden los mismos datos varias veces y no ocupar más espacio.

In [10]:
# CARGUE
guardar_db(dest_db_connection_string, ciudades, f'{db_destino}.Ciudad', db_user, db_psswd)

Verifique los resultados usando MySQL Workbench

### BLOQUE 3
Bloque 3: dimensión paquete. Su fuente de datos es la tabla transaccional <i>Paquetes</i>

#### Extracción

In [11]:
#EXTRACCION
sql_paquetes = '''(SELECT DISTINCT ID_TipoPaquete AS ID_TipoPaquete_T, TipoPaquete AS Nombre FROM WWImportersTransactional.Paquetes) AS Temp_Paquetes'''
paquetes = obtener_dataframe_de_bd(source_db_connection_string, sql_paquetes, db_user, db_psswd)

#### Transformación

In [12]:
# TRANSFORMACION
paquetes = paquetes.coalesce(1).withColumn('ID_TipoPaquete_DWH', f.monotonically_increasing_id() + 1)
paquetes = paquetes.select('ID_TipoPaquete_DWH','ID_TipoPaquete_T','Nombre')
paquetes.show(5)

+------------------+----------------+------+
|ID_TipoPaquete_DWH|ID_TipoPaquete_T|Nombre|
+------------------+----------------+------+
|                 1|               1|   Bag|
|                 2|               2| Block|
|                 3|               3|Bottle|
|                 4|               4|   Box|
|                 5|               5|   Can|
+------------------+----------------+------+
only showing top 5 rows



#### Carga

**OJO** Recuerde antes de guardar los datos que la tabla no exista o este vacía, para que no se guarden los mismos datos varias veces y no ocupar más espacio.

In [13]:
# CARGUE
guardar_db(dest_db_connection_string, paquetes,f'{db_destino}.TipoPaquete', db_user, db_psswd)

Verifique los resultados usando MySQL Workbench

### BLOQUE 4
Bloque 4: dimensión producto, su fuente de datos es la combinación entre las tablas transaccionales Productos y Colores

#### Extracción

In [14]:
sql_productos = '''(SELECT DISTINCT ID_Producto as ID_Producto_T, ID_Color, NombreProducto, Marca, Necesita_refrigeracion, Dias_tiempo_entrega, Impuesto, PrecioUnitario, PrecioRecomendado FROM WWImportersTransactional.Producto) AS Temp_productos'''
sql_colores = '''(SELECT DISTINCT ID_Color, Color FROM WWImportersTransactional.Colores) AS Temp_colores'''

productos = obtener_dataframe_de_bd(source_db_connection_string, sql_productos, db_user, db_psswd)
colores = obtener_dataframe_de_bd(source_db_connection_string, sql_colores, db_user, db_psswd)

#### Transformación

In [15]:
# TRANSFORMACION
productos = productos.join(colores, how = 'left', on = 'ID_Color').fillna({'Color': 'Missing'})
productos = productos.coalesce(1).withColumn('ID_Producto_DWH', f.monotonically_increasing_id() + 1)
productos = productos.select('ID_Producto_DWH','ID_Producto_T','NombreProducto','Marca','Color','Necesita_refrigeracion','Dias_tiempo_entrega','PrecioRecomendado','Impuesto','PrecioUnitario')
productos.show(5)

+---------------+-------------+--------------------+-----+----------+----------------------+-------------------+-----------------+--------+--------------+
|ID_Producto_DWH|ID_Producto_T|      NombreProducto|Marca|     Color|Necesita_refrigeracion|Dias_tiempo_entrega|PrecioRecomendado|Impuesto|PrecioUnitario|
+---------------+-------------+--------------------+-----+----------+----------------------+-------------------+-----------------+--------+--------------+
|              1|            1|USB missile launc...| NULL|   Missing|                     0|                 14|               37|      15|            25|
|              2|            2|USB rocket launch...| NULL|Steel Gray|                     0|                 14|               37|      15|            25|
|              3|            3|Office cube peris...| NULL|     Black|                     0|                 14|               28|      15|            19|
|              4|            4|USB food flash dr...| NULL|   Missing| 

#### Carga

**OJO** Recuerde antes de guardar los datos que la tabla no exista o este vacía, para que no se guarden los mismos datos varias veces y no ocupar más espacio.

In [16]:
# CARGUE
guardar_db(dest_db_connection_string, productos,f'{db_destino}.Producto', db_user, db_psswd)

Verifique los resultados usando MySQL Workbench

### BLOQUE 5
Bloque 5: dimensión cliente. Su fuente de datos es la combinación entre las tablas transaccionales Categorias de cliente, Grupos de compra y Clientes

#### Extracción

In [17]:
sql_categoriasCliente = '''(SELECT DISTINCT ID_Categoria, NombreCategoria FROM WWImportersTransactional.CategoriasCliente) AS Temp_categoriasclientes'''
sql_gruposCompra = '''(SELECT DISTINCT ID_GrupoCompra, NombreGrupoCompra FROM WWImportersTransactional.GruposCompra) AS Temp_gruposcompra'''
sql_clientes = '''(SELECT DISTINCT ID_Cliente as ID_Cliente_T, Nombre, ClienteFactura, ID_Categoria, ID_GrupoCompra, ID_CiudadEntrega, LimiteCredito, FechaAperturaCuenta, DiasPago FROM WWImportersTransactional.Clientes) AS Temp_clientes'''

categoriasCliente = obtener_dataframe_de_bd(source_db_connection_string, sql_categoriasCliente, db_user, db_psswd)
gruposCompra = obtener_dataframe_de_bd(source_db_connection_string, sql_gruposCompra, db_user, db_psswd)
clientes = obtener_dataframe_de_bd(source_db_connection_string, sql_clientes, db_user, db_psswd)

#### Transformación

In [18]:
# TRANSFORMACION 
# EL SUPUESTO QUE SE TIENE ES QUE TODOS LOS CLIENTES TIENEN TODOS SUS DATOS DE CATEGORIA Y GRUPO Y NO SE ESTÁN PERDIENDO CLIENTES AL REALIZAR ESTE JOIN
clientes = clientes.join(gruposCompra, how = 'left', on = 'ID_GrupoCompra')
clientes = clientes.alias('cl').join(categoriasCliente.alias('ct'), how = 'left', on = 'ID_Categoria') \
.select([col('cl.ID_Cliente_T'),col('cl.Nombre'),col('ct.NombreCategoria'),col('cl.NombreGrupoCompra') \
        ,col('cl.ClienteFactura'),col('cl.ID_CiudadEntrega'),col('cl.LimiteCredito'),col('cl.FechaAperturaCuenta'),col('cl.DiasPago')])
clientes = clientes.coalesce(1).withColumn('ID_Cliente_DWH', f.monotonically_increasing_id() + 1)
clientes = clientes.select('ID_Cliente_DWH','ID_Cliente_T','Nombre','NombreCategoria','NombreGrupoCompra','ClienteFactura',
                          'ID_CiudadEntrega','LimiteCredito','FechaAperturaCuenta','DiasPago')

clientes = clientes.fillna({'NombreCategoria':'Missing','NombreGrupoCompra':'Missing'})
clientes.show(5)

+--------------+------------+--------------------+---------------+-----------------+--------------+----------------+-------------+-------------------+--------+
|ID_Cliente_DWH|ID_Cliente_T|              Nombre|NombreCategoria|NombreGrupoCompra|ClienteFactura|ID_CiudadEntrega|LimiteCredito|FechaAperturaCuenta|DiasPago|
+--------------+------------+--------------------+---------------+-----------------+--------------+----------------+-------------+-------------------+--------+
|             1|           1|Tailspin Toys (He...|   Novelty Shop|    Tailspin Toys|             1|           19586|         NULL|2013-01-01 00:00:00|       7|
|             2|           2|Tailspin Toys (Sy...|   Novelty Shop|    Tailspin Toys|             1|           33475|         NULL|2013-01-01 00:00:00|       7|
|             3|           3|Tailspin Toys (Pe...|   Novelty Shop|    Tailspin Toys|             1|           26483|         NULL|2013-01-01 00:00:00|       7|
|             4|           4|Tailspin To

In [19]:
# Crea el registro para el id = 0
clientes_0 = [('0','','Missing','Missing','Missing','0','0','','','')]
columns = ['ID_Cliente_DWH','ID_Cliente_T','Nombre','NombreCategoria','NombreGrupoCompra','ClienteFactura',
            'ID_CiudadEntrega','LimiteCredito','FechaAperturaCuenta','DiasPago']
cliente_0 = spark.createDataFrame(data=clientes_0,schema=columns)
cliente_0.show()

+--------------+------------+-------+---------------+-----------------+--------------+----------------+-------------+-------------------+--------+
|ID_Cliente_DWH|ID_Cliente_T| Nombre|NombreCategoria|NombreGrupoCompra|ClienteFactura|ID_CiudadEntrega|LimiteCredito|FechaAperturaCuenta|DiasPago|
+--------------+------------+-------+---------------+-----------------+--------------+----------------+-------------+-------------------+--------+
|             0|            |Missing|        Missing|          Missing|             0|               0|             |                   |        |
+--------------+------------+-------+---------------+-----------------+--------------+----------------+-------------+-------------------+--------+



In [20]:
clientes = clientes.union(cliente_0)
clientes.show(5)

+--------------+------------+--------------------+---------------+-----------------+--------------+----------------+-------------+-------------------+--------+
|ID_Cliente_DWH|ID_Cliente_T|              Nombre|NombreCategoria|NombreGrupoCompra|ClienteFactura|ID_CiudadEntrega|LimiteCredito|FechaAperturaCuenta|DiasPago|
+--------------+------------+--------------------+---------------+-----------------+--------------+----------------+-------------+-------------------+--------+
|             1|           1|Tailspin Toys (He...|   Novelty Shop|    Tailspin Toys|             1|           19586|         NULL|2013-01-01 00:00:00|       7|
|             2|           2|Tailspin Toys (Sy...|   Novelty Shop|    Tailspin Toys|             1|           33475|         NULL|2013-01-01 00:00:00|       7|
|             3|           3|Tailspin Toys (Pe...|   Novelty Shop|    Tailspin Toys|             1|           26483|         NULL|2013-01-01 00:00:00|       7|
|             4|           4|Tailspin To

In [21]:
# Se ordena por el identificador DWH
clientes = clientes.withColumn('ID_Cliente_DWH',col('ID_Cliente_DWH').cast('int')).orderBy(col('ID_Cliente_DWH'))
clientes.show(5)

+--------------+------------+--------------------+---------------+-----------------+--------------+----------------+-------------+-------------------+--------+
|ID_Cliente_DWH|ID_Cliente_T|              Nombre|NombreCategoria|NombreGrupoCompra|ClienteFactura|ID_CiudadEntrega|LimiteCredito|FechaAperturaCuenta|DiasPago|
+--------------+------------+--------------------+---------------+-----------------+--------------+----------------+-------------+-------------------+--------+
|             0|            |             Missing|        Missing|          Missing|             0|               0|             |                   |        |
|             1|           1|Tailspin Toys (He...|   Novelty Shop|    Tailspin Toys|             1|           19586|         NULL|2013-01-01 00:00:00|       7|
|             2|           2|Tailspin Toys (Sy...|   Novelty Shop|    Tailspin Toys|             1|           33475|         NULL|2013-01-01 00:00:00|       7|
|             3|           3|Tailspin To

#### Carga

**OJO** Recuerde antes de guardar los datos que la tabla no exista o este vacía, para que no se guarden los mismos datos varias veces y no ocupar más espacio.

In [22]:
# CARGUE
guardar_db(dest_db_connection_string,clientes,f'{db_destino}.Cliente', db_user, db_psswd)

Verifique los resultados usando MySQL Workbench

### BLOQUE 6
Bloque 6: Hecho orden. Su fuente de datos es la combinación entre las tablas transaccionales Ordenes y detalles de orden

#### Extracción

In [23]:
sql_ordenes = '''(SELECT DISTINCT * FROM WWImportersTransactional.Ordenes) AS Temp_ordenes'''
sql_detallesOrdenes = '''(SELECT DISTINCT * FROM WWImportersTransactional.DetallesOrdenes) AS Temp_detallesordenes'''
ordenes = obtener_dataframe_de_bd(source_db_connection_string, sql_ordenes, db_user, db_psswd)
detallesOrdenes = obtener_dataframe_de_bd(source_db_connection_string, sql_detallesOrdenes, db_user, db_psswd)

#### Transformación
Estas son las respuestas de Wide World Importers a los conclusiones obtenidas en el entendimiento de los datos:
- La regla de negocio "La tasa de impuesto es de 10% o 15%" es correcta, pero habian errores en la tabla original, que fueron corregidos. 
- Para la segunda regla de negocio: "Son 73.595 órdenes detalladas en 231.412 lineas de detalle de órdenes realizadas desde 2013", si faltaban datos, los cuales fueron completados, y nos dicen que en cuanto a consistencia ellos revisaron las tablas e hicieron correcciones, pero que los duplicados completos de ordenes los eliminemos
- "El formato de fechas manejado es YYYY-MM-DD HH:MM:SS si tienen hora, minutos y segundos. De lo contrario el formato es YYYY-MM-DD": En cuanto a formatos de fechas estan de acuerdo con que los estandarizemos y el formato sea el especificado en la regla
- Para las descripciones de productos que eran "a", se actualizaron a los valores reales. 
- Se pueden eliminar las columnas Comenarios, Instrucciones_de_entrega y comentarios_internos porque estan vacias. 
- A pesar de estar en un proceso de mejorar la calidad de los datos y mantener los nulos nos ayudaría a reflejar esa calidad, de la mano con el grupo de analitica de WWI se decide imputar por la media el valor extremo de la variable Cantidad
- Para las ordenes las columnas Seleccionado_por_ID_de_persona, ID_de_pedido_pendiente, Seleccion_completada_cuando, y para las columnas Seleccion_completada_cuando de la tabla detalles de ordenes, se decide mantener los valores vacíos, sin embargo para la variable Precio_unitario el negocio reviso y complemento los valores faltantes

Las tablas usadas en el tutorial de entendimiento de datos estaran disponibles para su revision con los siguientes nombres: OrdenesCopia y DetallesOrdenesCopia. 

Para este tutorial vamos a trabajar con unas tablas que dadas las conclusiones del tutorial de entendimiento, WWImporters revisó los datos originales, creo tablas y las llamo "Ordenes" y "DetallesOrdenes"

Se hace una verificación de los valores de la tasa de impuesto

In [24]:
detallesOrdenes.select("Tasa_de_impuesto").distinct().show()

+----------------+
|Tasa_de_impuesto|
+----------------+
|              15|
|              10|
+----------------+



Se hace una verificación del rango de fechas disponible en los datos

In [25]:
ordenes.agg({"Fecha_de_pedido": "min"}).show()

+--------------------+
|min(Fecha_de_pedido)|
+--------------------+
|          2013-01-01|
+--------------------+



Se elimina columnas Comenarios, Instrucciones_de_entrega y comentarios_internos

In [26]:
ordenes = ordenes.drop(*["Comentarios", "Instrucciones_de_entrega","comentarios_internos"])

Se eliminan duplicados totales de ordenes

In [27]:
print((ordenes.count(),ordenes.distinct().count()))

(93629, 93629)


In [28]:
ordenes = ordenes.drop_duplicates()

In [29]:
print((ordenes.count(),ordenes.distinct().count()))

(93629, 93629)


Se hace verificación de consistencia

In [30]:
#consistencia: revisar genially: definicion de consistencia
ids_ordenes = set([x.ID_de_pedido for x in ordenes.select('ID_de_pedido').collect()])
ids_detalles = set([x.ID_de_pedido for x in detallesOrdenes.select('ID_de_pedido').collect()])

len(ids_ordenes-ids_detalles), len(ids_detalles-ids_ordenes)

(0, 0)

En el siguiente código para el manejo de fechas, pasamos del formato MM dd,YYYY al formato establecido en la regla de negocio<br>

In [31]:
# TRANSFORMACION
regex = r"([0-2]\d{3}-(0[1-9]|1[0-2])-(0[1-9]|[1-2][0-9]|3[0-1]))"
cumplenFormato = ordenes.filter(ordenes["Fecha_de_pedido"].rlike(regex))
noCumplenFormato = ordenes.filter(~ordenes["Fecha_de_pedido"].rlike(regex))
print(noCumplenFormato.count(), cumplenFormato.count())
noCumplenFormato.show(5)
noCumplenFormato = noCumplenFormato.withColumn('Fecha_de_pedido', f.udf(lambda d: datetime.strptime(d, '%b %d,%Y').strftime('%Y-%m-%d'), t.StringType())(f.col('Fecha_de_pedido')))
ordenes = noCumplenFormato.union(cumplenFormato)
noCumplenFormato.count(), ordenes.count()

20034 73595
+------------+-------------+--------------+------------------------------+-------------------------+----------------------+---------------+-------------------------+--------------------------------------+-------------------------------------------+---------------------------+
|ID_de_pedido|ID_de_cliente|ID_de_vendedor|Seleccionado_por_ID_de_persona|ID_de_persona_de_contacto|ID_de_pedido_pendiente|Fecha_de_pedido|Fecha_de_entrega_esperada|Numero_de_pedido_de_compra_del_cliente|Pedido_pendiente_de_suministro_insuficiente|Seleccion_completada_cuando|
+------------+-------------+--------------+------------------------------+-------------------------+----------------------+---------------+-------------------------+--------------------------------------+-------------------------------------------+---------------------------+
|       19455|           86|            16|                             7|                     1171|                  NULL|    Jan 01,2014|               201

(20034, 93629)

Descripciones


In [32]:
detallesOrdenes.where(length(col("Descripcion")) <= 10).show()

+----------------+------------+-----------+-----------+---------------+--------+---------------+----------------+---------------------+---------------------------+
|Detalle_orden_ID|ID_de_pedido|ID_Producto|Descripcion|ID_Tipo_Paquete|Cantidad|Precio_unitario|Tasa_de_impuesto|Cantidad_seleccionada|Seleccion_completada_cuando|
+----------------+------------+-----------+-----------+---------------+--------+---------------+----------------+---------------------+---------------------------+
+----------------+------------+-----------+-----------+---------------+--------+---------------+----------------+---------------------+---------------------------+



Imputar valor maximo de cantidad

In [33]:
detallesOrdenes.select('Cantidad').sort(col("Cantidad").desc()).collect()[1]

Row(Cantidad=360)

In [34]:
detallesOrdenes = detallesOrdenes.replace( 10000000, 360, 'Cantidad')

In [35]:
detallesOrdenes.select('Cantidad').sort(col("Cantidad").desc()).collect()[0]

Row(Cantidad=360)

In [36]:
detallesOrdenes.show(5)

+----------------+------------+-----------+--------------------+---------------+--------+---------------+----------------+---------------------+---------------------------+
|Detalle_orden_ID|ID_de_pedido|ID_Producto|         Descripcion|ID_Tipo_Paquete|Cantidad|Precio_unitario|Tasa_de_impuesto|Cantidad_seleccionada|Seleccion_completada_cuando|
+----------------+------------+-----------+--------------------+---------------+--------+---------------+----------------+---------------------+---------------------------+
|               1|          45|        164|32 mm Double side...|              7|      50|            112|              15|                   50|        2013-01-02 11:00:00|
|               2|           1|         67|Ride on toy sedan...|              7|      10|            230|              15|                   10|        2013-01-01 11:00:00|
|               3|           2|         50|Developer joke mu...|              7|       9|             13|              15|             

In [37]:
ordenes.show(5)

+------------+-------------+--------------+------------------------------+-------------------------+----------------------+---------------+-------------------------+--------------------------------------+-------------------------------------------+---------------------------+
|ID_de_pedido|ID_de_cliente|ID_de_vendedor|Seleccionado_por_ID_de_persona|ID_de_persona_de_contacto|ID_de_pedido_pendiente|Fecha_de_pedido|Fecha_de_entrega_esperada|Numero_de_pedido_de_compra_del_cliente|Pedido_pendiente_de_suministro_insuficiente|Seleccion_completada_cuando|
+------------+-------------+--------------+------------------------------+-------------------------+----------------------+---------------+-------------------------+--------------------------------------+-------------------------------------------+---------------------------+
|       19455|           86|            16|                             7|                     1171|                  NULL|     2014-01-01|               2014-01-02|    

Se unen los dos dataframes en un nuevo dataframe, se verifica que no haya duplicados y si los hay se eliminan. Se crea un nuevo dataframe que va a tener toda la información del hecho orden transformada y lista para continuar el proceso de cargue.

In [38]:
ordenes_tmp =ordenes
ordenes_tmp = ordenes_tmp.join(detallesOrdenes, how = 'inner', on = 'ID_de_pedido')
ordenes_tmp = ordenes_tmp.withColumn('Valor_total',col('Precio_unitario')*col('Cantidad'))
ordenes_tmp = ordenes_tmp.withColumn('Impuestos',col('Valor_total')*col('Tasa_de_impuesto')/100)   # la tasa está en porcentaje
ordenes_tmp = ordenes_tmp.selectExpr('ID_de_pedido as ID_de_pedido_T','ID_Producto','Fecha_de_pedido','ID_de_cliente','ID_de_vendedor','ID_Tipo_Paquete','Cantidad','Valor_total', 'Impuestos')

print((ordenes_tmp.count(),ordenes_tmp.distinct().count()))

ordenes_tmp = ordenes_tmp.drop_duplicates()
ordenes_tmp.show(5)

(294314, 231412)
+--------------+-----------+---------------+-------------+--------------+---------------+--------+-----------+---------+
|ID_de_pedido_T|ID_Producto|Fecha_de_pedido|ID_de_cliente|ID_de_vendedor|ID_Tipo_Paquete|Cantidad|Valor_total|Impuestos|
+--------------+-----------+---------------+-------------+--------------+---------------+--------+-----------+---------+
|             2|         50|     2013-01-01|          803|             8|              7|       9|        117|17.550000|
|             3|        114|     2013-01-01|          105|             7|              7|       3|         90|13.500000|
|             4|        206|     2013-01-01|           57|            16|              7|      96|        288|43.200000|
|             6|        150|     2013-01-01|          976|            13|              9|       4|         64| 9.600000|
|            55|        200|     2013-01-01|          473|            13|              7|      96|        384|57.600000|
+--------------

In [39]:
guardar_db(dest_db_connection_string, ordenes_tmp,f'{db_destino}.Hecho_Orden_Tmp', db_user, db_psswd)

Cree la tabla de Fecha según el material compartido y adicione el left join al crear la tabla de ordenes para que quede completa

In [40]:
# El idPedido representa la dimensión degenerada Pedido
# Si hay campos nulos en ordenes_tmp al hacer join por el left outer join no se perderan y se utiliza como comodín un id=0 que debe existir en todas las dimensiones.
# Ese comodín representa el registro sin Dato.
# Debe adicionarle a todas las tablas el registro con identificador 0, como se muestra para la tabla de clientes
# Recuerde que falta incluir el join con la tabla de Fecha para que la tabla quede completa.

ordenes = ordenes_tmp.alias('o').join(clientes.alias('cl'), ordenes_tmp.ID_de_cliente == clientes.ID_Cliente_T,'left')\
                    .join(ciudades.alias('ciu'), clientes.ID_CiudadEntrega == ciudades.ID_ciudad_T,'left') \
                    .join(empleados.alias('e'), ordenes_tmp.ID_de_vendedor == empleados.ID_Empleado_T,'left') \
                    .join(paquetes.alias('p'), ordenes_tmp.ID_Tipo_Paquete == paquetes.ID_TipoPaquete_T,'left') \
                    .join(productos.alias('pr'), (ordenes_tmp.ID_Producto == productos.ID_Producto_T) ,'left') \
                    .select([col('o.ID_de_pedido_T'),col('cl.ID_Cliente_DWH'),col('ciu.ID_Ciudad_DWH'),
                             col('e.ID_Empleado_DWH'),col('pr.ID_Producto_DWH'),col('p.ID_TipoPaquete_DWH'),
                             col('o.Cantidad'),col('o.Valor_total'),col('o.Impuestos')]) \
                    .fillna({'ID_Cliente_DWH': 0, 'ID_Ciudad_DWH': 0, 'ID_Empleado_DWH': 0, 'ID_Producto_DWH': 0,
                             'ID_TipoPaquete_DWH': 0})
ordenes.show(5)

+--------------+--------------+-------------+---------------+---------------+------------------+--------+-----------+---------+
|ID_de_pedido_T|ID_Cliente_DWH|ID_Ciudad_DWH|ID_Empleado_DWH|ID_Producto_DWH|ID_TipoPaquete_DWH|Cantidad|Valor_total|Impuestos|
+--------------+--------------+-------------+---------------+---------------+------------------+--------+-----------+---------+
|             2|           405|        33588|              5|             50|                 7|       9|        117|17.550000|
|             3|           105|        33996|              4|            114|                 7|       3|         90|13.500000|
|             4|            57|        18417|              9|            206|                 7|      96|        288|43.200000|
|             6|           578|        30713|              6|            150|                 9|       4|         64| 9.600000|
|            55|           274|        11687|              6|            200|                 7|      96

In [41]:
ordenes = ordenes.select('ID_de_pedido_T','ID_Ciudad_DWH','ID_Cliente_DWH','ID_Empleado_DWH','ID_Producto_DWH',
                         'ID_TipoPaquete_DWH','Cantidad','Impuestos','Valor_total') \
                    .withColumnRenamed('Valor_total','Total')
ordenes.show(5)

+--------------+-------------+--------------+---------------+---------------+------------------+--------+---------+-----+
|ID_de_pedido_T|ID_Ciudad_DWH|ID_Cliente_DWH|ID_Empleado_DWH|ID_Producto_DWH|ID_TipoPaquete_DWH|Cantidad|Impuestos|Total|
+--------------+-------------+--------------+---------------+---------------+------------------+--------+---------+-----+
|             2|        33588|           405|              5|             50|                 7|       9|17.550000|  117|
|             3|        33996|           105|              4|            114|                 7|       3|13.500000|   90|
|             4|        18417|            57|              9|            206|                 7|      96|43.200000|  288|
|             6|        30713|           578|              6|            150|                 9|       4| 9.600000|   64|
|            55|        11687|           274|              6|            200|                 7|      96|57.600000|  384|
+--------------+--------

#### Carga

**OJO** Recuerde antes de guardar los datos que la tabla no exista o este vacía, para que no se guarden los mismos datos varias veces y no ocupar más espacio.

In [42]:
guardar_db(dest_db_connection_string, ordenes,f'{db_destino}.Hecho_Orden', db_user, db_psswd)

Verifique los resultados usando MySQL Workbench

# Resultado de consultas
Corresponde a las consultas realizadas sobre las tablas, para mostrar el estado final de las tablas pobladas como resultado del proceso de ETL.

# 3. Tarea ETL
Espacio para desarrollar la tarea planteada

### Bloque 1: Dimensión Proveedor

#### _Extracción_

In [9]:
# Extracción de proveedores de la base transaccional
sql_proveedores = '''(
    SELECT DISTINCT 
        ProveedorID, 
        NombreProveedor, 
        CategoriaProveedorID, 
        PersonaContactoPrincipalID, 
        DiasPago, 
        CodigoPostal
    FROM WWImportersTransactional.proveedoresCopia
) AS Temp_proveedores'''

proveedores_raw = obtener_dataframe_de_bd(source_db_connection_string, sql_proveedores, 
                                          db_user, db_psswd)

# Extraccion de las categorias de proveedor
sql_categorias = '''(
    SELECT DISTINCT 
        CategoriaProveedorID, 
        CategoriaProveedor
    FROM WWImportersTransactional.CategoriasProveedores
) AS Temp_cat_proveedor'''

categorias_proveedor = obtener_dataframe_de_bd(source_db_connection_string, 
                                               sql_categorias, db_user, db_psswd)

# Extracción de Personas
sql_personas = '''(
    SELECT DISTINCT 
        ID_persona, 
        NombreCompleto AS ContactoPrincipal
    FROM WWImportersTransactional.Personas
) AS Temp_personas_contacto'''

personas_contacto = obtener_dataframe_de_bd(source_db_connection_string, 
                                            sql_personas, db_user, db_psswd)

print("--- Muestra de Proveedores Extraídos ---")
proveedores_raw.show(5)

--- Muestra de Proveedores Extraídos ---
+-----------+--------------------+--------------------+--------------------------+--------+------------+
|ProveedorID|     NombreProveedor|CategoriaProveedorID|PersonaContactoPrincipalID|DiasPago|CodigoPostal|
+-----------+--------------------+--------------------+--------------------------+--------+------------+
|          4|       Fabrikam Inc.|                   4|                        27|      30|       40351|
|          5|Graphic Design In...|                   2|                        29|      14|       64847|
|          7|        Litware Inc.|                   5|                        33|      30|       95245|
|          9|      Nod Publishers|                   2|                        37|       7|       27906|
|         10|Northwind Electri...|                   3|                        39|      30|        7860|
+-----------+--------------------+--------------------+--------------------------+--------+------------+
only showing t

#### _Transformación:_

T1. Deduplicación: Se descartan duplicados por ProveedorID.

T2. Cruce (Left Joins): Se cruza con categorías (CategoriaProveedorID) y con personas (PersonaContactoPrincipalID).

T3. Regla de Negocio WWI (Días de pago negativos): Si DiasPago < 0, se corrige multiplicando por -1.

T4. Imputación de Nulos: Se asigna 'Sin Categoría' y 'Sin Contacto' a registros sin cruce.

T5. Renombrar al esquema DWH: Adaptar nombres a ID_Proveedor_T, Nombre, Categoria, Contacto_principal, Dias_pago, Codigo_postal.

T6. Llave Subrogada: Generar ID_Proveedor_DWH mediante monotonically_increasing_id() + 1.

T7. Registro Comodín (ID=0): Se inserta la fila con ID_Proveedor_DWH = 0 para soportar transacciones donde no participa un proveedor (ej. ventas a clientes).


In [10]:
# T1. Deduplicación
proveedores = proveedores_raw.dropDuplicates(['ProveedorID'])

# T2. Cruce
proveedores = proveedores.join(categorias_proveedor, 
                               on='CategoriaProveedorID', how='left')
proveedores = proveedores.join(personas_contacto, 
                               col('PersonaContactoPrincipalID') == col('ID_persona'), how='left')

# T3. Días de pago negativos
proveedores = proveedores.withColumn(
    'DiasPagoCorregido',
    when(col('DiasPago') < 0, col('DiasPago') * -1).otherwise(col('DiasPago'))
)

# T4. Imputación de Nulos
proveedores = proveedores.fillna({
    'CategoriaProveedor': 'Sin Categoría',
    'ContactoPrincipal': 'Sin Contacto',
    'CodigoPostal': '00000'
})

# T5. Renombrar al esquema DWH
proveedores = proveedores.select(
    col('ProveedorID').cast('int').alias('ID_Proveedor_T'),
    col('NombreProveedor').alias('Nombre'),
    col('CategoriaProveedor').alias('Categoria'),
    col('ContactoPrincipal').alias('Contacto_principal'),
    col('DiasPagoCorregido').cast('int').alias('Dias_pago'),
    col('CodigoPostal').alias('Codigo_postal')
)

# T6. Llave Subrogada
proveedores = proveedores.coalesce(1).withColumn('ID_Proveedor_DWH', 
                                                 (f.monotonically_increasing_id() + 1).cast('int'))

# T7. Registro Comodín ID = 0
columns_orden = ['ID_Proveedor_DWH', 'ID_Proveedor_T', 'Nombre', 'Categoria', 
                 'Contacto_principal', 'Dias_pago', 'Codigo_postal']

proveedor_comodin_data = [(0, 0, 'No Aplica / Sin Proveedor', 'No Aplica', 'No Aplica', 0, '00000')]
df_proveedor_comodin = spark.createDataFrame(
    data=proveedor_comodin_data,
    schema=columns_orden
)

# Unir y ordenar por ID_Proveedor_DWH ascendente
proveedores = proveedores.select(columns_orden).union(df_proveedor_comodin)
proveedores = proveedores.orderBy('ID_Proveedor_DWH')

print(f"Total registros listos en Dimensión Proveedor: {proveedores.count()}")
proveedores.show(10, truncate=False)

Total registros listos en Dimensión Proveedor: 20
+----------------+--------------+-------------------------+-----------------------+------------------+---------+-------------+
|ID_Proveedor_DWH|ID_Proveedor_T|Nombre                   |Categoria              |Contacto_principal|Dias_pago|Codigo_postal|
+----------------+--------------+-------------------------+-----------------------+------------------+---------+-------------+
|0               |0             |No Aplica / Sin Proveedor|No Aplica              |No Aplica         |0        |00000        |
|1               |1             |A Datum Corporation      |productos novedosos    |Reio Kabin        |14       |46077        |
|2               |2             |Contoso Ltd.             |productos novedosos    |Hanna Mihhailov   |7        |98253        |
|3               |3             |Consolidated Messenger   |servicios de mensajeria|Kerstin Parn      |30       |94101        |
|4               |4             |Fabrikam Inc.            |ro

#### Carga
Se realiza el cargue en la tabla correspondiente de la bodega de datos relacional.

In [11]:
guardar_db(dest_db_connection_string, proveedores, f'{db_destino}.Proveedor', db_user, db_psswd)
print("¡Carga exitosa en la base de datos!")

sql_verificacion = f"(SELECT * FROM {db_destino}.Proveedor) AS Temp_verif_prov"
proveedores_bd = obtener_dataframe_de_bd(dest_db_connection_string, sql_verificacion, db_user, db_psswd)
print(f"Total registros verificados en MySQL ({db_destino}.Proveedor): {proveedores_bd.count()}")
proveedores_bd.show(5)

¡Carga exitosa en la base de datos!
Total registros verificados en MySQL (DB_202615_df_perezj.Proveedor): 40
+----------------+--------------+--------------------+--------------------+------------------+---------+-------------+
|ID_Proveedor_DWH|ID_Proveedor_T|              Nombre|           Categoria|Contacto_principal|Dias_pago|Codigo_postal|
+----------------+--------------+--------------------+--------------------+------------------+---------+-------------+
|               0|             0|No Aplica / Sin P...|           No Aplica|         No Aplica|        0|        00000|
|               1|             1| A Datum Corporation| productos novedosos|        Reio Kabin|       14|        46077|
|               2|             2|        Contoso Ltd.| productos novedosos|   Hanna Mihhailov|        7|        98253|
|               3|             3|Consolidated Mess...|servicios de mens...|      Kerstin Parn|       30|        94101|
|               4|             4|       Fabrikam Inc.|    

### Bloque 2: Dimensión TipoTransacción

_Extracción_

In [12]:
# Extracción de transacción de la base transaccional
sql_tipos_transaccion = '''(
    SELECT DISTINCT 
        TipoTransaccionID, 
        TipoTransaccionNombre 
    FROM WWImportersTransactional.TiposTransaccion
) AS Temp_tipos_transaccion'''
tipos_transaccion_raw = obtener_dataframe_de_bd(source_db_connection_string, 
                                                sql_tipos_transaccion, db_user, db_psswd)
print("--- Muestra de Tipos de Transacción Extraídos ---")
tipos_transaccion_raw.show(10, truncate=False)

--- Muestra de Tipos de Transacción Extraídos ---
+-----------------+-------------------------+
|TipoTransaccionID|TipoTransaccionNombre    |
+-----------------+-------------------------+
|2                |Customer Credit Note     |
|3                |Customer Payment Received|
|4                |Customer Refund          |
|5                |Supplier Invoice         |
|6                |Supplier Credit Note     |
|7                |Supplier Payment Issued  |
|8                |Supplier Refund          |
|9                |Stock Transfer           |
|10               |Stock Issue              |
|11               |Stock Receipt            |
+-----------------+-------------------------+
only showing top 10 rows



#### _Transformación:_

T1. Descarte de duplicados: Garantizar unicidad por TipoTransaccionID.

T2. Estandarización y renombrado: Ajustar nombres al modelo dimensional (ID_Tipo_transaccion_T y Tipo).

T3. Llave subrogada DWH: Generar ID_Tipo_transaccion_DWH con monotonically_increasing_id() + 1.

T4. Registro comodín (ID=0): Insertar el registro artificial con llave 0 para transacciones con tipo no tipificado.

In [13]:
# T1. Descarte de duplicados
tipos_transaccion = tipos_transaccion_raw.dropDuplicates(['TipoTransaccionID'])

# T2. Estandarización y renombrado
tipos_transaccion = tipos_transaccion.select(
    col('TipoTransaccionID').cast('int').alias('ID_Tipo_transaccion_T'),
    col('TipoTransaccionNombre').alias('Tipo')
)

# T3. Llave subrogada DWH
tipos_transaccion = tipos_transaccion.coalesce(1).withColumn(
    'ID_Tipo_transaccion_DWH', 
    (f.monotonically_increasing_id() + 1).cast('int')
)

# Registro comodín ID=0
columns_orden_tt = ['ID_Tipo_transaccion_DWH', 'ID_Tipo_transaccion_T', 'Tipo']

tipo_comodin_data = [(0, 0, 'No Aplica / Desconocido')]
df_tipo_comodin = spark.createDataFrame(
    data=tipo_comodin_data, 
    schema=columns_orden_tt
)

# Unir comodín y ordenar por la llave subrogada
tipos_transaccion = tipos_transaccion.select(columns_orden_tt).union(df_tipo_comodin)
tipos_transaccion = tipos_transaccion.orderBy('ID_Tipo_transaccion_DWH')

print(f"Total registros listos en Dimensión TipoTransacción: {tipos_transaccion.count()}")
tipos_transaccion.show(10, truncate=False)

Total registros listos en Dimensión TipoTransacción: 13
+-----------------------+---------------------+-------------------------+
|ID_Tipo_transaccion_DWH|ID_Tipo_transaccion_T|Tipo                     |
+-----------------------+---------------------+-------------------------+
|0                      |0                    |No Aplica / Desconocido  |
|1                      |2                    |Customer Credit Note     |
|2                      |3                    |Customer Payment Received|
|3                      |4                    |Customer Refund          |
|4                      |5                    |Supplier Invoice         |
|5                      |6                    |Supplier Credit Note     |
|6                      |7                    |Supplier Payment Issued  |
|7                      |8                    |Supplier Refund          |
|8                      |9                    |Stock Transfer           |
|9                      |10                   |Stock Iss

#### Carga


Se realiza el cargue en la tabla TipoTransaccion de la bodega de datos relacional.

In [14]:
guardar_db(dest_db_connection_string, tipos_transaccion, f'{db_destino}.TipoTransaccion', 
           db_user, db_psswd)
print("¡Carga exitosa en la base de datos!")

sql_verificacion_tt = f"(SELECT * FROM {db_destino}.TipoTransaccion) AS Temp_verif_tt"
tt_bd = obtener_dataframe_de_bd(dest_db_connection_string, sql_verificacion_tt, db_user, db_psswd)
print(f"Total registros verificados en MySQL ({db_destino}.TipoTransaccion): {tt_bd.count()}")
tt_bd.show(5)

¡Carga exitosa en la base de datos!
Total registros verificados en MySQL (DB_202615_df_perezj.TipoTransaccion): 26
+-----------------------+---------------------+--------------------+
|ID_Tipo_transaccion_DWH|ID_Tipo_transaccion_T|                Tipo|
+-----------------------+---------------------+--------------------+
|                      0|                    0|No Aplica / Desco...|
|                      1|                    2|Customer Credit Note|
|                      2|                    3|Customer Payment ...|
|                      3|                    4|     Customer Refund|
|                      4|                    5|    Supplier Invoice|
+-----------------------+---------------------+--------------------+
only showing top 5 rows



### Bloque 3: Dimensión Fecha

#### _Extracción_

In [15]:
# Extracción de fechas de las transacciones
sql_fechas = '''(
    SELECT DISTINCT 
        FechaTransaccion
    FROM WWImportersTransactional.movimientosCopia
    WHERE FechaTransaccion IS NOT NULL
) AS Temp_fechas'''

fechas_raw = obtener_dataframe_de_bd(source_db_connection_string, sql_fechas, db_user, db_psswd)
print(f"Total marcas temporales extraídas: {fechas_raw.count()}")
fechas_raw.show(5, truncate=False)

Total marcas temporales extraídas: 2155
+----------------+
|FechaTransaccion|
+----------------+
|Apr 25,2014     |
|Dec 10,2015     |
|Dec 04,2015     |
|Dec 23,2015     |
|Jul 27,2015     |
+----------------+
only showing top 5 rows



#### _Transformación:_

T1. Extracción de fechas únicas: Descartar duplicados a nivel de día calendario en el rango histórico 2013–2016.

T2. Normalización de formato: Convertir los múltiples formatos (yyyy-MM-dd HH:mm:ss, MMM dd,yyyy, etc.) al tipo nativo DATE (YYYY-MM-DD).

T3. Llave entera ID_Fecha: Generar el identificador numérico de negocio en formato YYYYMMDD (ej. 20140425).

T4. Jerarquía temporal: Calcular atributos derivados de análisis (Dia, Mes, Anio, Numero_semana_ISO).

T5. Registro comodín (ID=0): Insertar la fila comodín ID_Fecha = 0 con fecha base (1900-01-01) para transacciones con fecha no identificada.

In [16]:
# La IA me ayudo a encontrar la solución a un problema que tiene que ver con el analizador de fechas
# de java 8 por que al leer las fechas me arrojaba valoes con microsegundos entonces me dijo que 
# al actualizar la politica de spark a legacy para poder leer la fecha ignorando los microsegundos 
# si la fecha tiene - pues solo se toman los 10 primeros caracteres

# Configurar política de parser legacy para soportar múltiples formatos sin abortar
spark.conf.set("spark.sql.legacy.timeParserPolicy", "LEGACY")

# T2. Normalización de formato
fechas_limpias = fechas_raw.withColumn(
    'Fecha',
    when(
        col('FechaTransaccion').contains('-'),
        f.to_date(f.substring(col('FechaTransaccion'), 1, 10), 'yyyy-MM-dd')
    ).otherwise(
        f.coalesce(
            f.to_date(f.trim(col('FechaTransaccion')), 'MMM dd,yyyy'),
            f.to_date(f.trim(col('FechaTransaccion')), 'MMM d,yyyy'),
            f.to_date(f.trim(col('FechaTransaccion')), 'MMM dd, yyyy'),
            f.to_date(f.trim(col('FechaTransaccion')), 'MMM d, yyyy')
        )
    )
).filter(col('Fecha').isNotNull())

# T1. Extracción de fechas únicas
fechas = fechas_limpias.select('Fecha').dropDuplicates(['Fecha'])

# T3 y T4. Llave entera ID_Fecha y Jerarquía temporal
fechas = fechas.select(
    f.date_format(col('Fecha'), 'yyyyMMdd').cast('int').alias('ID_Fecha'),
    col('Fecha'),
    f.dayofmonth(col('Fecha')).cast('int').alias('Dia'),
    f.month(col('Fecha')).cast('int').alias('Mes'),
    f.year(col('Fecha')).cast('int').alias('Anio'),
    f.weekofyear(col('Fecha')).cast('int').alias('Numero_semana_ISO')
)

# T5. Registro comodín ID_Fecha = 0
columns_orden_fecha = ['ID_Fecha', 'Fecha', 'Dia', 'Mes', 'Anio', 'Numero_semana_ISO']

# Usamos '1900-01-01' como fecha base no nula para respetar la restricción NOT NULL de MySQL
fecha_comodin_data = [(0, '1900-01-01', 0, 0, 0, 0)]
df_fecha_comodin = spark.createDataFrame(
    data=fecha_comodin_data, 
    schema=['ID_Fecha', 'Fecha_str', 'Dia', 'Mes', 'Anio', 'Numero_semana_ISO']
).select(
    col('ID_Fecha').cast('int'),
    f.to_date(col('Fecha_str'), 'yyyy-MM-dd').alias('Fecha'),
    col('Dia').cast('int'),
    col('Mes').cast('int'),
    col('Anio').cast('int'),
    col('Numero_semana_ISO').cast('int')
)

fechas = fechas.select(columns_orden_fecha).union(df_fecha_comodin)
fechas = fechas.orderBy('ID_Fecha')

print(f"Total días en la Dimensión Fecha: {fechas.count()}")
fechas.show(10, truncate=False)

Total días en la Dimensión Fecha: 759
+--------+----------+---+---+----+-----------------+
|ID_Fecha|Fecha     |Dia|Mes|Anio|Numero_semana_ISO|
+--------+----------+---+---+----+-----------------+
|0       |1900-01-01|0  |0  |0   |0                |
|20131231|2013-12-31|31 |12 |2013|1                |
|20140101|2014-01-01|1  |1  |2014|1                |
|20140102|2014-01-02|2  |1  |2014|1                |
|20140103|2014-01-03|3  |1  |2014|1                |
|20140104|2014-01-04|4  |1  |2014|1                |
|20140106|2014-01-06|6  |1  |2014|2                |
|20140107|2014-01-07|7  |1  |2014|2                |
|20140108|2014-01-08|8  |1  |2014|2                |
|20140109|2014-01-09|9  |1  |2014|2                |
+--------+----------+---+---+----+-----------------+
only showing top 10 rows



#### Carga

Se realiza el cargue en la tabla M_Fecha de la bodega de datos relacional.

In [17]:
guardar_db(dest_db_connection_string, fechas, f'{db_destino}.Fecha', db_user, db_psswd)
print("¡Carga exitosa en la base de datos!")

sql_verificacion_fechas = f"(SELECT * FROM {db_destino}.Fecha) AS Temp_verif_fecha"
fechas_bd = obtener_dataframe_de_bd(dest_db_connection_string, sql_verificacion_fechas, 
                                    db_user, db_psswd)

print(f"Total registros verificados en MySQL ({db_destino}.Fecha): {fechas_bd.count()}")
fechas_bd.show(5)

¡Carga exitosa en la base de datos!
Total registros verificados en MySQL (DB_202615_df_perezj.Fecha): 1518
+--------+----------+---+---+----+-----------------+
|ID_Fecha|     Fecha|Dia|Mes|Anio|Numero_semana_ISO|
+--------+----------+---+---+----+-----------------+
|       0|1900-01-01|  0|  0|   0|                0|
|20131231|2013-12-31| 31| 12|2013|                1|
|20140101|2014-01-01|  1|  1|2014|                1|
|20140102|2014-01-02|  2|  1|2014|                1|
|20140103|2014-01-03|  3|  1|2014|                1|
+--------+----------+---+---+----+-----------------+
only showing top 5 rows



### Bloque 4: Tabla de Hechos Hecho_Movimiento

#### _Extracción_

In [18]:
# 1. Extracción de Movimientos Transaccionales
sql_movimientos = '''(
    SELECT 
        TransaccionProductoID,
        ProductoID,
        TipoTransaccionID,
        ClienteID,
        ProveedorID,
        FechaTransaccion,
        Cantidad
    FROM WWImportersTransactional.movimientosCopia
) AS Temp_movimientos'''
movimientos_raw = obtener_dataframe_de_bd(source_db_connection_string, sql_movimientos, db_user, db_psswd)
print(f"Total movimientos extraídos: {movimientos_raw.count()}")
movimientos_raw.show(5)

Total movimientos extraídos: 204292
+---------------------+----------+-----------------+---------+-----------+----------------+--------+
|TransaccionProductoID|ProductoID|TipoTransaccionID|ClienteID|ProveedorID|FechaTransaccion|Cantidad|
+---------------------+----------+-----------------+---------+-----------+----------------+--------+
|               118903|       217|               10|    476.0|           |     Apr 25,2014|   -40.0|
|               286890|       135|               10|     33.0|           |     Dec 10,2015|    -7.0|
|               285233|       111|               10|    180.0|           |     Dec 04,2015|    -2.0|
|               290145|       213|               10|     33.0|           |     Dec 23,2015|    -3.0|
|               247492|        90|               10|     55.0|           |     Jul 27,2015|   -24.0|
+---------------------+----------+-----------------+---------+-----------+----------------+--------+
only showing top 5 rows



#### _Transformación:_

T1. Descarte de duplicados: Deduplicación por TransaccionProductoID.

T2. Validación de métrica Cantidad: Verificación de no nulidad, respetando el sentido del signo (positivo = entrada, negativo = salida) y omitiendo cualquier tope de 50M (regla de negocio WWI).

T3. Búsqueda de llaves foráneas (Lookups): Cruce con las 5 dimensiones del modelo (Fecha, Producto, Proveedor, Cliente, TipoTransaccion).

T4. Tratamiento de llaves nulas con comodín (ID=0): Imputación de llaves foráneas faltantes con 0 (fundamental para ventas que no tienen proveedor y compras que no tienen cliente).

T5. Proyección y tipado final: Selección estricta de las 5 llaves foráneas y la métrica Cantidad.

In [23]:
# Configuración de compatibilidad para el analizador de fechas
spark.conf.set("spark.sql.legacy.timeParserPolicy", "LEGACY")

# T1. Descarte de duplicados por TransaccionProductoID
movimientos = movimientos_raw.dropDuplicates(['TransaccionProductoID'])

# T2. Validación de Cantidad y casteo de llaves
movimientos = movimientos.filter(col('Cantidad').isNotNull()) \
    .withColumn('ProductoID', col('ProductoID').cast('int')) \
    .withColumn('ProveedorID', col('ProveedorID').cast('int')) \
    .withColumn('ClienteID', col('ClienteID').cast('int')) \
    .withColumn('TipoTransaccionID', col('TipoTransaccionID').cast('int'))

# Dimensión Fecha: Normalización y generación de ID_Fecha (YYYYMMDD)
movimientos = movimientos.withColumn(
    'Fecha_Norm',
    when(
        col('FechaTransaccion').contains('-'),
        f.to_date(f.substring(col('FechaTransaccion'), 1, 10), 'yyyy-MM-dd')
    ).otherwise(
        f.coalesce(
            f.to_date(f.trim(col('FechaTransaccion')), 'MMM dd,yyyy'),
            f.to_date(f.trim(col('FechaTransaccion')), 'MMM d,yyyy'),
            f.to_date(f.trim(col('FechaTransaccion')), 'MMM dd, yyyy')
        )
    )
).withColumn(
    'ID_Fecha',
    f.coalesce(f.date_format(col('Fecha_Norm'), 'yyyyMMdd').cast('int'), f.lit(0))
)

# Cargar llaves subrogadas de dimensiones (nombres correctos sin M_)
dim_producto = obtener_dataframe_de_bd(
    dest_db_connection_string, 
    f"(SELECT ID_Producto_T, ID_Producto_DWH FROM {db_destino}.Producto) AS Temp_p", 
    db_user, db_psswd
).select(col('ID_Producto_T').cast('int'), col('ID_Producto_DWH').cast('int')).dropDuplicates(['ID_Producto_T'])

dim_cliente = obtener_dataframe_de_bd(
    dest_db_connection_string, 
    f"(SELECT ID_Cliente_T, ID_Cliente_DWH FROM {db_destino}.Cliente) AS Temp_c", 
    db_user, db_psswd
).select(col('ID_Cliente_T').cast('int'), col('ID_Cliente_DWH').cast('int')).dropDuplicates(['ID_Cliente_T'])

dim_proveedor = obtener_dataframe_de_bd(
    dest_db_connection_string, 
    f"(SELECT ID_Proveedor_T, ID_Proveedor_DWH FROM {db_destino}.Proveedor) AS Temp_prov", 
    db_user, db_psswd
).select(col('ID_Proveedor_T').cast('int'), col('ID_Proveedor_DWH').cast('int')).dropDuplicates(['ID_Proveedor_T'])

dim_tipo = obtener_dataframe_de_bd(
    dest_db_connection_string, 
    f"(SELECT ID_Tipo_transaccion_T, ID_Tipo_transaccion_DWH FROM {db_destino}.TipoTransaccion) AS Temp_tipo", 
    db_user, db_psswd
).select(col('ID_Tipo_transaccion_T').cast('int'), col('ID_Tipo_transaccion_DWH').cast('int')).dropDuplicates(['ID_Tipo_transaccion_T'])

# T3. Cruces (Left Joins) estrictamente 1 a 1
movimientos = movimientos.join(
    dim_producto,
    movimientos.ProductoID == dim_producto.ID_Producto_T,
    how='left'
).join(
    dim_proveedor,
    movimientos.ProveedorID == dim_proveedor.ID_Proveedor_T,
    how='left'
).join(
    dim_cliente,
    movimientos.ClienteID == dim_cliente.ID_Cliente_T,
    how='left'
).join(
    dim_tipo,
    movimientos.TipoTransaccionID == dim_tipo.ID_Tipo_transaccion_T,
    how='left'
)

# T4. Asignación del comodín ID = 0 a llaves foráneas nulas
movimientos = movimientos.fillna({
    'ID_Fecha': 0,
    'ID_Producto_DWH': 0,
    'ID_Proveedor_DWH': 0,
    'ID_Cliente_DWH': 0,
    'ID_Tipo_transaccion_DWH': 0
})

# T5. Proyección y tipado final
hecho_movimiento = movimientos.select(
    col('ID_Fecha').cast('int'),
    col('ID_Producto_DWH').cast('int'),
    col('ID_Proveedor_DWH').cast('int'),
    col('ID_Cliente_DWH').cast('int'),
    col('ID_Tipo_transaccion_DWH').cast('int'),
    col('Cantidad').cast('int')
)

print(f"Total registros listos en Hecho_Movimiento: {hecho_movimiento.count()}")
hecho_movimiento.show(10)

Total registros listos en Hecho_Movimiento: 173659
+--------+---------------+----------------+--------------+-----------------------+--------+
|ID_Fecha|ID_Producto_DWH|ID_Proveedor_DWH|ID_Cliente_DWH|ID_Tipo_transaccion_DWH|Cantidad|
+--------+---------------+----------------+--------------+-----------------------+--------+
|20131231|             77|               4|             0|                     10|   10188|
|20131231|             95|               4|             0|                     10|    6444|
|20131231|             98|               4|             0|                     10|   12672|
|20131231|            204|               7|             0|                     10|    5140|
|20131231|             86|               0|             2|                      9|     -60|
|20131231|            175|               0|           365|                      9|      -2|
|20131231|            173|               0|           201|                      9|     -10|
|20131231|            164|   

#### Carga

Cargue masivo en la tabla de hechos de la bodega de datos relacional.

In [25]:
guardar_db(dest_db_connection_string, hecho_movimiento, f'{db_destino}.Hecho_Movimiento', 
           db_user, db_psswd)
print("¡Carga exitosa en la base de datos!")


sql_verificacion_hm = f"(SELECT * FROM {db_destino}.Hecho_Movimiento) AS Temp_verif_hm"
hm_bd = obtener_dataframe_de_bd(dest_db_connection_string, sql_verificacion_hm, db_user, db_psswd)


print(f"Total registros verificados en MySQL ({db_destino}.Hecho_Movimiento): {hm_bd.count()}")
hm_bd.show(10)

¡Carga exitosa en la base de datos!
Total registros verificados en MySQL (DB_202615_df_perezj.Hecho_Movimiento): 173659
+--------+---------------+----------------+--------------+-----------------------+--------+
|ID_Fecha|ID_Producto_DWH|ID_Proveedor_DWH|ID_Cliente_DWH|ID_Tipo_transaccion_DWH|Cantidad|
+--------+---------------+----------------+--------------+-----------------------+--------+
|20131231|             80|               4|             0|                     10|    4824|
|20131231|             86|               4|             0|                     10|   10368|
|20131231|            193|               7|             0|                     10|   10248|
|20131231|             77|               0|           368|                      9|     -60|
|20131231|             14|               0|           365|                      9|      -1|
|20131231|              2|               0|           201|                      9|      -6|
|20131231|             16|               0|         

## 4. Cierre
Completado este tutorial, ya sabe cómo realizar ETL básicos en PySpark.


## 5. Información adicional

Si quiere conocer más sobre PySpark la guía más detallada es la documentación oficial, la cual puede encontrar acá: https://spark.apache.org/docs/latest/api/python/index.html <br>
Para ir directamente a la documentación de PySpark SQL, donde está la información sobre los DataFrames, haga clic en este enlace: https://spark.apache.org/docs/latest/api/python/pyspark.sql.html <br>

El Capítulo 2 del libro <i>Learn PySpark : Build Python-based Machine Learning and Deep Learning Models, New York: Apress. 2019</i> de Pramod Singh contiene muchos ejemplos útiles, puede encontrarlo en la biblioteca virtual de la universidad.

## 6. Preguntas frecuentes

- Si al intentar escribir un <i>dataframe</i> obtiene un error en el formato: 
    ```
    path file:<PATH>/dw/<PATH> already exists.;
    ```
    Borre la carpeta indicada en el error y vuelva a intentar.

- Si aparece el error <i>PYTHON_VERSION_MISMATCH</i>, reinicie el kernel y ejecute el notebook desde el inicio.

- Los mensajes <i>NativeCodeLoader: Unable to load native-hadoop library</i>, <i>winutils</i> o <i>Using incubator modules: jdk.incubator.vector</i> son advertencias, no errores.

In [43]:
spark.stop()